In [2]:
%pip install requests beautifulsoup4 pandas
%pip install yfinance
%pip install transformers torch

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
# ========================================================
# TAHAP 1: DATA COLLECTION 
# ========================================================
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os
from IPython.display import display

FOLDER_BTC = r"D:\Collage\AnuJurnal\dataset\datasetbtc"
FOLDER_FED = r"D:\Collage\AnuJurnal\dataset\teksgacor_powelthefed"

# Memastikan kedua folder tersebut dibuat secara otomatis jika belum ada
os.makedirs(FOLDER_BTC, exist_ok=True)
os.makedirs(FOLDER_FED, exist_ok=True)

print(" MEMULAI TAHAP 1: DATA COLLECTION...")
print("-" * 50)

 MEMULAI TAHAP 1: DATA COLLECTION...
--------------------------------------------------


In [4]:
# ========================================================
# BAGIAN A: MENGUNDUH DATA BITCOIN (Dari 2020-01-01 hingga sekarang)
# ========================================================
print("1. Mengunduh data historis Bitcoin...")
btc_data = yf.download('BTC-USD', start="2020-01-01") 

# Merapikan format tabel Bitcoin
if isinstance(btc_data.columns, pd.MultiIndex):
    btc_data.columns = btc_data.columns.get_level_values(0)
btc_data.index = pd.to_datetime(btc_data.index).tz_localize(None)

# Menyimpan ke folder spesifik BTC
path_btc = os.path.join(FOLDER_BTC, "01_raw_btc.csv")
btc_data.to_csv(path_btc, sep=';', decimal=',')
print(f" Data BTC tersimpan: {len(btc_data)} hari.")
print(f" Lokasi BTC: {path_btc}")
print("-" * 50)

1. Mengunduh data historis Bitcoin...


[*********************100%***********************]  1 of 1 completed

 Data BTC tersimpan: 2318 hari.
 Lokasi BTC: D:\Collage\AnuJurnal\dataset\datasetbtc\01_raw_btc.csv
--------------------------------------------------


In [5]:

# --------------------------------------------------------
# BAGIAN B: SCRAPING TEKS  THE FED (MASSAL)
# --------------------------------------------------------
daftar_target_fomc = [
    # 2010
    {"tanggal": "2010-02-17", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20100127.htm"},
    {"tanggal": "2010-04-06", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20100316.htm"},
    {"tanggal": "2010-05-19", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20100428.htm"},
    {"tanggal": "2010-07-14", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20100623.htm"},
    {"tanggal": "2010-08-31", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20100810.htm"},
    {"tanggal": "2010-10-12", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20100921.htm"},
    {"tanggal": "2010-11-23", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20101103.htm"},
    {"tanggal": "2011-01-04", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20101214.htm"},

    # 2011
    {"tanggal": "2011-02-16", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20110126.htm"},
    {"tanggal": "2011-04-05", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20110315.htm"},
    {"tanggal": "2011-05-18", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20110427.htm"},
    {"tanggal": "2011-07-12", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20110622.htm"},
    {"tanggal": "2011-08-30", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20110809.htm"},
    {"tanggal": "2011-10-12", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20110921.htm"},
    {"tanggal": "2011-11-22", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20111102.htm"},
    {"tanggal": "2012-01-03", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20111213.htm"},

    # 2012
    {"tanggal": "2012-02-15", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20120125.htm"},
    {"tanggal": "2012-04-03", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20120313.htm"},
    {"tanggal": "2012-05-16", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20120425.htm"},
    {"tanggal": "2012-07-11", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20120620.htm"},
    {"tanggal": "2012-08-22", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20120801.htm"},
    {"tanggal": "2012-10-04", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20120913.htm"},
    {"tanggal": "2012-11-14", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20121024.htm"},
    {"tanggal": "2013-01-03", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20121212.htm"},

    # 2013
    {"tanggal": "2013-02-20", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20130130.htm"},
    {"tanggal": "2013-04-10", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20130320.htm"},
    {"tanggal": "2013-05-22", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20130501.htm"},
    {"tanggal": "2013-07-10", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20130619.htm"},
    {"tanggal": "2013-08-21", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20130731.htm"},
    {"tanggal": "2013-10-09", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20130918.htm"},
    {"tanggal": "2013-11-20", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20131030.htm"},
    {"tanggal": "2014-01-08", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20131218.htm"},

    # 2014
    {"tanggal": "2014-02-19", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20140129.htm"},
    {"tanggal": "2014-04-09", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20140319.htm"},
    {"tanggal": "2014-05-21", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20140430.htm"},
    {"tanggal": "2014-07-09", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20140618.htm"},
    {"tanggal": "2014-08-20", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20140730.htm"},
    {"tanggal": "2014-10-08", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20140917.htm"},
    {"tanggal": "2014-11-19", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20141029.htm"},
    {"tanggal": "2015-01-07", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20141217.htm"},

    # 2015
    {"tanggal": "2015-02-18", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20150128.htm"},
    {"tanggal": "2015-04-08", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20150318.htm"},
    {"tanggal": "2015-05-20", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20150429.htm"},
    {"tanggal": "2015-07-08", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20150617.htm"},
    {"tanggal": "2015-08-19", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20150729.htm"},
    {"tanggal": "2015-10-08", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20150917.htm"},
    {"tanggal": "2015-11-18", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20151028.htm"},
    {"tanggal": "2016-01-06", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20151216.htm"},

    # 2016
    {"tanggal": "2016-02-17", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20160127.htm"},
    {"tanggal": "2016-04-06", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20160316.htm"},
    {"tanggal": "2016-05-18", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20160427.htm"},
    {"tanggal": "2016-07-06", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20160615.htm"},
    {"tanggal": "2016-08-17", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20160727.htm"},
    {"tanggal": "2016-10-12", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20160921.htm"},
    {"tanggal": "2016-11-23", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20161102.htm"},
    {"tanggal": "2017-01-04", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20161214.htm"},

    # 2017
    {"tanggal": "2017-02-22", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20170201.htm"},
    {"tanggal": "2017-04-05", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20170315.htm"},
    {"tanggal": "2017-05-24", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20170503.htm"},
    {"tanggal": "2017-07-05", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20170614.htm"},
    {"tanggal": "2017-08-16", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20170726.htm"},
    {"tanggal": "2017-10-11", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20170920.htm"},
    {"tanggal": "2017-11-22", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20171101.htm"},
    {"tanggal": "2018-01-03", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20171213.htm"},

    # 2018
    {"tanggal": "2018-02-21", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20180131.htm"},
    {"tanggal": "2018-04-11", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20180321.htm"},
    {"tanggal": "2018-05-23", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20180502.htm"},
    {"tanggal": "2018-07-05", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20180613.htm"},
    {"tanggal": "2018-08-22", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20180801.htm"},
    {"tanggal": "2018-10-17", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20180926.htm"},
    {"tanggal": "2018-11-29", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20181108.htm"},
    {"tanggal": "2019-01-09", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20181219.htm"},

    # 2019
    {"tanggal": "2019-02-20", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20190130.htm"},
    {"tanggal": "2019-04-10", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20190320.htm"},
    {"tanggal": "2019-05-22", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20190501.htm"},
    {"tanggal": "2019-07-10", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20190619.htm"},
    {"tanggal": "2019-08-21", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20190731.htm"},
    {"tanggal": "2019-10-09", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20190918.htm"},
    {"tanggal": "2019-11-20", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20191030.htm"},
    {"tanggal": "2020-01-03", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20191211.htm"},
    #2020
    {"tanggal": "2020-02-19", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20200129.htm"},
    {"tanggal": "2020-04-08", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20200315.htm"},
    {"tanggal": "2020-05-20", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20200429.htm"},
    {"tanggal": "2020-07-01", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20200610.htm"},
    {"tanggal": "2020-08-19", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20200729.htm"},
    {"tanggal": "2020-10-07", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20200916.htm"},
    {"tanggal": "2020-11-25", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20201105.htm"},
    {"tanggal": "2021-01-06", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20201216.htm"},
     #2021
    {"tanggal": "2021-02-17", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20210127.htm"},#2021
    {"tanggal": "2021-04-07", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20210317.htm"},
    {"tanggal": "2021-05-19", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20210428.htm"},
    {"tanggal": "2021-07-07", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20210616.htm"},
    {"tanggal": "2021-08-18", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20210728.htm"},
    {"tanggal": "2021-10-13", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20210922.htm"},
    {"tanggal": "2022-01-05", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20211215.htm"}, 
    #2022 
    {"tanggal": "2022-02-16", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20220126.htm"},
    {"tanggal": "2022-04-06", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20220316.htm"},
    {"tanggal": "2022-05-25", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20220504.htm"},
    {"tanggal": "2022-07-06", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20220615.htm"},
    {"tanggal": "2022-08-17", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20220727.htm"},
    {"tanggal": "2022-10-12", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20220921.htm"},
    {"tanggal": "2022-11-23", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20221102.htm"},
    {"tanggal": "2023-01-04", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20221214.htm"}, 
    #2023
    {"tanggal": "2023-02-22", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20230201.htm"},
    {"tanggal": "2023-04-12", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20230322.htm"},
    {"tanggal": "2023-05-24", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20230503.htm"},
    {"tanggal": "2023-07-05", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20230614.htm"},
    {"tanggal": "2023-10-11", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20230920.htm"},
    {"tanggal": "2023-11-21", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20231101.htm"},
    {"tanggal": "2024-01-03", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20231213.htm"},
    #2024
    {"tanggal": "2024-02-21", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20240131.htm"},
    {"tanggal": "2024-04-10", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20240320.htm"},
    {"tanggal": "2024-05-22", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20240501.htm"},
    {"tanggal": "2024-07-03", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20240612.htm"},
    {"tanggal": "2024-08-21", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20240731.htm"},
    {"tanggal": "2024-10-09", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20240918.htm"},
    {"tanggal": "2024-11-26", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20241107.htm"},
    {"tanggal": "2025-01-08", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20241218.htm"},
    #2025
    {"tanggal": "2025-02-19", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20250129.htm"},
    {"tanggal": "2025-04-09", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20250319.htm"},
    {"tanggal": "2025-05-08", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20250507.htm"},
    {"tanggal": "2025-07-09", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20250618.htm"},
    {"tanggal": "2025-08-20", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20250730.htm"},
    {"tanggal": "2025-10-08", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20250917.htm"},
    {"tanggal": "2025-11-19", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20251029.htm"},
    {"tanggal": "2025-12-30", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20251210.htm"},
    #2026
    {"tanggal": "2026-02-18", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20260128.htm"},
    {"tanggal": "2026-04-08", "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20260318.htm"}
]

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9'
}

data_thefed = []
print(f"2. Memulai ekstraksi {len(daftar_target_fomc)} dokumen The Fed...")

# Proses Looping Web Scraping
for item in daftar_target_fomc:
    tanggal = item["tanggal"]
    url = item["url"]
    print(f"   Menyedot tanggal {tanggal}...", end=" ")
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            main_div = soup.find('div', id='article') or soup.find('div', id='content')
            paragraphs = main_div.find_all('p') if main_div else soup.find_all('p')
            
            full_text = " ".join([p.get_text(strip=True) for p in paragraphs])
            
            if len(full_text) > 300:
                data_thefed.append({'Tanggal': tanggal, 'Teks_Pidato': full_text})
                print(f" Sukses ({len(full_text)} karakter)")
            else:
                print(" Gagal (Teks kosong/terblokir)")
        else:
            print(f" Gagal HTTP {response.status_code}")
    except Exception as e:
        print(f" Error: {e}")
        
    time.sleep(2) # Istirahat 2 detik

2. Memulai ekstraksi 128 dokumen The Fed...
   Menyedot tanggal 2010-02-17...  Sukses (68260 karakter)
   Menyedot tanggal 2010-04-06...  Sukses (42026 karakter)
   Menyedot tanggal 2010-05-19...  Sukses (44588 karakter)
   Menyedot tanggal 2010-07-14...  Sukses (50903 karakter)
   Menyedot tanggal 2010-08-31...  Sukses (42011 karakter)
   Menyedot tanggal 2010-10-12...  Sukses (38073 karakter)
   Menyedot tanggal 2010-11-23...  Sukses (46120 karakter)
   Menyedot tanggal 2011-01-04...  Sukses (42696 karakter)
   Menyedot tanggal 2011-02-16...  Sukses (66557 karakter)
   Menyedot tanggal 2011-04-05...  Sukses (43446 karakter)
   Menyedot tanggal 2011-05-18...  Sukses (49633 karakter)
   Menyedot tanggal 2011-07-12...  Sukses (45160 karakter)
   Menyedot tanggal 2011-08-30...  Sukses (42585 karakter)
   Menyedot tanggal 2011-10-12...  Sukses (52881 karakter)
   Menyedot tanggal 2011-11-22...  Sukses (48919 karakter)
   Menyedot tanggal 2012-01-03...  Sukses (49176 karakter)
   Menyedot 